# Notebook 3: answer questions on a Neo4j graph with neo4j-graphrag

[neo4j-graphrag](https://github.com/neo4j/neo4j-graphrag-python) is Neo4j's Python library for GraphRAG. A
**retriever** finds the facts in the graph that answer a question, and an LLM writes the answer from those facts.

We added decision models to the library in a
[pull request](https://github.com/neo4j/neo4j-graphrag-python/pull/728). It adds two classes:

- `OpenRouterDecisionModel` calls a decision model on OpenRouter.
- `DecisionWalkRetriever` walks the graph with a decision model, as in notebook 1, and returns the nodes that the walk
  ends on.

The pull request is not merged yet, so `pip install neo4j-graphrag` does not have these classes. This repository
installs the pull request's version of the library (see `pyproject.toml`), so it is already in your environment. To
use it in another project, install it with:

```
pip install "neo4j-graphrag[openai,openrouter] @ https://github.com/shamalinnodatagit/neo4j-graphrag-python/archive/a1b2abf55738491bd476d342c04f47f5af9ed3ad.zip"
```

In this notebook you will:

1. make one decision with `OpenRouterDecisionModel`,
2. answer questions with `DecisionWalkRetriever`, and read every decision of the walk,
3. turn the walk into a written answer with `GraphRAG`,
4. send each question to the retriever that suits it, with `ToolsRetriever`,
5. try your own questions, and see what to change for your own graph.

You need `OPENROUTER_API_KEY` in the `.env` file. Running the whole notebook costs less than one cent.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # so the notebook can import utils/ from the repository root

import neo4j
import pandas as pd
from neo4j_graphrag.decision import OpenRouterDecisionModel
from neo4j_graphrag.retrievers import DecisionWalkRetriever

from utils import display
from utils.settings import openrouter_key

OPENROUTER_API_KEY = openrouter_key()  # read from the .env file

## The graph

We use Neo4j's public `recommendations` demo database. It holds movies, the people who acted in and directed them,
their genres, and ratings by users. It is read-only and online, so there is nothing to load.

To use your own graph, change these settings. The last section says what each one needs.

In [ ]:
URI = "neo4j+s://demo.neo4jlabs.com"
AUTH = ("recommendations", "recommendations")
DATABASE = "recommendations"
FULLTEXT_INDEX = "personFulltext"     # a full-text index on the names of the nodes that questions start from
NAME_PROPERTIES = ["name", "title"]   # the node properties that hold a node's name, in order

driver = neo4j.GraphDatabase.driver(URI, auth=AUTH)
records, _, _ = driver.execute_query(
    "MATCH ()-[r]->() WHERE NOT type(r) STARTS WITH '_' RETURN type(r) AS relationship, count(*) AS count "
    "ORDER BY count DESC", database_=DATABASE)
pd.DataFrame([r.data() for r in records])

These relationship types are what a walk chooses between. For example, `ACTED_IN` goes from a person to a movie, and
`DIRECTED` goes from a director to a movie.

## 1. One decision

At each step of a walk, the retriever writes a numbered list of options: one for each relationship type and direction
around the current nodes, and STOP. A decision model reads the question and the options, and it returns its choice and
a probability for every option.

![One step of the walk: the retriever writes the options, and the decision model gives each one a probability](images/walk_one_step.svg)

In the library, one decision is one call: `decide(state, options, instructions)`. Here is the first step of "Who
directed the movies that Tom Hanks acted in?", written by hand.

In [ ]:
decision_model = OpenRouterDecisionModel("liquid/d1-ultra", api_key=OPENROUTER_API_KEY)

response = decision_model.decide(
    state={"question": "Who directed the movies that Tom Hanks acted in?", "current_entities": ["Tom Hanks"]},
    options={
        "0": "STOP: the answer is: Tom Hanks",
        "1": "follow ACTED_IN (outgoing) to: Apollo 13, Big, Cast Away, Forrest Gump ...",
        "2": "follow DIRECTED (outgoing) to: Larry Crowne, That Thing You Do!",
    },
    instructions="Choose the connection to follow next to answer the question.",
)
print(f"Chosen option: {response.choice}   cost: ${response.usage.cost:.6f}   model: {response.model}")
pd.Series(response.probabilities, name="probability")

## 2. Walk the graph with `DecisionWalkRetriever`

The retriever makes these decisions for you, one step at a time:

1. **Start.** It searches the full-text index for the question, and the decision model chooses the node that the
   question is about.
2. **Step.** It groups the relationships of the nodes it stands on into options, one for each relationship type and
   direction. Option 0 is always STOP. The decision model chooses an option, and the walk follows it to all of its
   nodes.
3. **Stop.** The walk ends when the model chooses STOP, or after `max_steps` steps (3 by default).

The search returns the nodes that the walk ended on. Each one comes with the path that reached it. Here is the whole
walk for the Tom Hanks question, which you run later in this section:

![The whole walk: the start node and three steps, each chosen by the decision model](images/walk_whole.svg)

First, a question that needs only one step.

In [ ]:
retriever = DecisionWalkRetriever(
    driver=driver,
    decision_model=decision_model,
    fulltext_index_name=FULLTEXT_INDEX,
    node_name_properties=NAME_PROPERTIES,
    neo4j_database=DATABASE,
)

result = retriever.search(query_text="Which movies did Hugo Weaving act in?")
for item in result.items:
    print(item.metadata["path"])

`result.metadata` describes the walk: where it started, each step, why it stopped, and what the decisions cost.

In [ ]:
walk = result.metadata
print("Started at: ", ", ".join(walk["start_nodes"]))
print("Steps:      ", *walk["path"], sep="\n  ")
print("Stopped by: ", walk["stopped_by"])
print(f"Nodes found: {walk['result_nodes_total']}")
print(f"Decisions:   {walk['decision_calls']}, for ${walk['decision_cost']:.6f} in total")

### Every decision of the walk

`metadata["decisions"]` keeps every decision: the options, the probability of each, and the choice. The first
decision chose the start node from the results of the full-text search. The table shows the top 4 options of each
decision.

In [ ]:
display.decision_log(walk)

### What the LLM will read

Each result is one line of text: the node's name and labels, the path that reached it, and its properties. This is
what an LLM reads when it writes the answer in section 3.

In [ ]:
print(result.items[0].content[:600])

### A longer walk, from a node you choose

If you already know the node that a question is about, for example because a user clicked on it, pass its element id
as `start_node_ids`. The walk then starts there, without the full-text search.

In [ ]:
records, _, _ = driver.execute_query(
    "MATCH (p:Person {name: 'Tom Hanks'}) RETURN elementId(p) AS id", database_=DATABASE)
result = retriever.search(query_text="Who directed the movies that Tom Hanks acted in?",
                          start_node_ids=[records[0]["id"]])

print(*result.metadata["path"], sep="\n")
print(f"\n{result.metadata['result_nodes_total']} directors. The first five, with their paths:")
for item in result.items[:5]:
    print("  ", item.metadata["path"])
display.decision_log(result.metadata)

## 3. A written answer with `GraphRAG`

`GraphRAG` joins a retriever and an LLM. The retriever finds the nodes, and the LLM writes the answer from them. The
LLM does not search the graph; it only writes.

![From the walk to a written answer: the retriever walks the graph, GraphRAG puts the results into a prompt, and the LLM writes the answer](images/graphrag_answer.svg)

The LLM here is GPT-5.6 Luna on OpenRouter. We use the library's `OpenAILLM` class, because OpenRouter accepts the
same requests as OpenAI.

In [ ]:
from neo4j_graphrag.generation import GraphRAG
from neo4j_graphrag.llm import OpenAILLM

llm = OpenAILLM(
    model_name="openai/gpt-5.6-luna",
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    model_params={"extra_body": {"reasoning": {"effort": "none"}}},  # answer without reasoning
)
rag = GraphRAG(retriever=retriever, llm=llm)

answer = rag.search(query_text="What genres are the movies that Christopher Nolan directed?", return_context=True)
print("Walk:", *answer.retriever_result.metadata["path"], sep="\n  ")
print("\nAnswer:", answer.answer)

## 4. Questions that a walk can't answer

A walk follows relationships from a named node. It can't count, average, rank, or filter by property values, so it
can't answer "How many movies did Tom Hanks act in?". The library's `Text2CypherRetriever` can: an LLM writes a Cypher
query, and Neo4j runs it.

The `ToolsRetriever` lets an LLM choose between them. You give it both retrievers as **tools**, each with a
description of the questions it is for, and the LLM picks a tool for each question.

![Choosing a retriever for each question: an LLM sends the Hugo Weaving question to the walk and the counting question to Text2Cypher](images/routing.svg)

Each tool below also prints what it did: the steps of the walk, where `>` follows a relationship forwards and `<`
follows it backwards, or the Cypher query.

In [ ]:
from neo4j_graphrag.retrievers import Text2CypherRetriever, ToolsRetriever
from neo4j_graphrag.tool import ObjectParameter, StringParameter, Tool

text2cypher = Text2CypherRetriever(driver, llm, neo4j_database=DATABASE)


def follow_relationships(query_text: str):
    result = retriever.search(query_text=query_text)
    print("   the walk:", " -> ".join(result.metadata["steps"]))
    return result


def query_database(query_text: str):
    result = text2cypher.search(query_text=query_text)
    print("   the Cypher query:", " ".join(result.metadata["cypher"].split()))
    return result


QUESTION_ONLY = ObjectParameter(
    description="Parameters for the search",
    properties={"query_text": StringParameter(description="The user's question")},
    required_properties=["query_text"],
    additional_properties=False,
)
tools = [
    Tool(
        name="follow_relationships",
        description=(
            "Finds things connected to a specific, named person by following relationships in the movie graph, one "
            "or more steps. Use it for questions that start from a named person, such as 'Which movies did Hugo "
            "Weaving act in?' or 'Who directed the movies Tom Hanks acted in?'. Do not use it for counting, "
            "averages, rankings, sorting, or filtering by numbers or dates."
        ),
        parameters=QUESTION_ONLY,
        execute_func=follow_relationships,
    ),
    Tool(
        name="query_database",
        description=(
            "Writes and runs a database query. Use it for questions that count, average, rank, sort or filter by "
            "property values, or that don't start from one named entity, such as 'How many movies did Tom Hanks act "
            "in?' or 'What are the 5 highest-rated movies?'."
        ),
        parameters=QUESTION_ONLY,
        execute_func=query_database,
    ),
]
router = GraphRAG(retriever=ToolsRetriever(driver=driver, llm=llm, tools=tools), llm=llm)

for question in ["Who directed the movies that Hugo Weaving acted in?", "How many movies did Tom Hanks act in?"]:
    print(f"\nQ: {question}")
    answer = router.search(query_text=question, return_context=True)
    print("   tool:", ", ".join(answer.retriever_result.metadata["tools_selected"]))
    print("A:", answer.answer)

## 5. Your turn

Change the question and run the cell. The full-text index here holds people's names, so start your questions from a
person. Some ideas:

- "Who acted in the movies that Christopher Nolan directed?"
- "Which genres are the movies Meryl Streep acted in?"
- "Who else acted in the movies Keanu Reeves acted in?"

To start from movies instead, set `FULLTEXT_INDEX = "movieFulltext"` and run the cells again from the top.

In [ ]:
MY_QUESTION = "Who acted in the movies that Christopher Nolan directed?"

answer = rag.search(query_text=MY_QUESTION, return_context=True)
print("Walk:", *answer.retriever_result.metadata["path"], sep="\n  ")
print("\nAnswer:", answer.answer)

### On your own graph

The retriever works on any Neo4j graph. Change the settings at the top of the notebook:

1. `URI`, `AUTH` and `DATABASE`: your database.
2. `FULLTEXT_INDEX`: a full-text index on the names of the nodes that questions start from. If you don't have one,
   create it once. For example:

   ```cypher
   CREATE FULLTEXT INDEX entityNames IF NOT EXISTS FOR (n:Person|Company|Product) ON EACH [n.name]
   ```

3. `NAME_PROPERTIES`: the properties that hold a node's name, in order.

The decision model reads the relationship types to choose each step. Clear names such as `ACTED_IN` or `WORKS_FOR`
help it more than a general name such as `RELATED_TO`. For walks of more than 3 steps, pass `max_steps` to the
retriever.

Close the connections when you are done. In a notebook, use `await decision_model.aclose()`; in a script, use `decision_model.close()`.

In [ ]:
await decision_model.aclose()
driver.close()